In [1]:
from google.colab import drive
drive.mount("/content/drive")
import importlib
import subprocess
import sys
from huggingface_hub import login
from google.colab import userdata

login(userdata.get("HF_TOKEN"))


Mounted at /content/drive


In [2]:
#!pip install -q numpy==1.26.4
!pip install -q transformers==4.44.2
!pip install -q transformer_lens==2.11.0
!pip install -q datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
plum-dispatch 2.8.0 requires beartype>=0.16.2, but you have beartype 0.14.1 which is incompatible.


In [3]:
from datasets import load_dataset, interleave_datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
class SAE(nn.Module):
    def __init__(self, d_in, d_hidden, l1_coeff=3e-4):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_hidden, bias=False)
        self.decoder = nn.Linear(d_hidden, d_in, bias=False)
        self.l1_coeff = l1_coeff

        # Tied weights option (recommended for interpretability)
        self.decoder.weight = nn.Parameter(self.encoder.weight.T)

    def forward(self, x):
        # Sparse latent features
        z = F.relu(self.encoder(x))
        # Reconstruction
        x_hat = self.decoder(z)
        return x_hat, z

    def loss(self, x, x_hat, z, step=None):
      mse = torch.mean((x_hat - x) ** 2)

      # 🔑 L1 warmup (critical)
      if step is not None and step < 500:
          return mse

      l1 = z.abs().mean()
      return mse + self.l1_coeff * l1

def load_pile_streaming():
    """The Pile — just grab the 'text' field directly."""
    ds = load_dataset(
    "monology/pile-uncopyrighted",
    split="train",
    streaming=True,
)

    return ds.map(lambda x: {"text": x["text"]})


def load_rewardbench_streaming():
    """
    RewardBench has prompt/chosen/rejected triples.
    We want BOTH chosen and rejected as separate text samples
    so the SAE sees reward-relevant response distributions.
    """
    ds = load_dataset(
        "allenai/reward-bench",
        streaming=True,
        split="filtered",
        trust_remote_code=True
    )

    def format_rewardbench(example):
        # Yield chosen and rejected as separate texts
        # We concatenate prompt + response so the SAE sees full context
        chosen_text  = f"{example['prompt']}\n\n{example['chosen']}"
        rejected_text = f"{example['prompt']}\n\n{example['rejected']}"
        # Return chosen — rejected will be handled via flat_map below
        return {"text": chosen_text, "_rejected": rejected_text}

    return ds.map(format_rewardbench)


import json
import requests
from datasets import IterableDataset

def load_sycophancy_streaming():
    SUBSETS = ["answer", "are_you_sure", "feedback", "mimicry"]
    BASE_URL = "https://huggingface.co/datasets/meg-tong/sycophancy-eval/resolve/main"

    def generate_examples():
        for subset in SUBSETS:
            url = f"{BASE_URL}/{subset}.jsonl"
            response = requests.get(url, stream=True)
            for line in response.iter_lines():
                if not line:
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    continue

                messages = row["prompt"]
                text = "\n\n".join(
                    f"{'Human' if m['type'] == 'human' else 'Assistant'}: {m['content']}"
                    for m in messages
                )
                text += "\n\nAssistant:"
                yield {"text": text, "subset": subset}

    return IterableDataset.from_generator(generate_examples)

In [4]:
def load_ultrafeedback_streaming():
    ds = load_dataset(
        "HuggingFaceH4/ultrafeedback_binarized",
        streaming=True,
        split="train_prefs",
    )
    ds = ds.shuffle(buffer_size=10_000, seed=None)

    def format_ultrafeedback(example):
        # chosen is a list of message dicts — extract the assistant response
        chosen_messages = example["chosen"]
        prompt    = chosen_messages[0]["content"]  # user turn
        response  = chosen_messages[1]["content"]  # assistant turn
        return {"text": f"{prompt}\n\n{response}"}

    return ds.map(format_ultrafeedback)


def build_combined_dataset(
    pile_weight=0.50,
    rewardbench_weight=0.20,
    sycophancy_weight=0.15,
    ultrafeedback_weight=0.15,
    seed=42
):
    pile          = load_pile_streaming()
    rewardbench   = load_rewardbench_streaming()
    sycophancy    = load_sycophancy_streaming()
    ultrafeedback = load_ultrafeedback_streaming()

    pile          = pile.select_columns(["text"]).repeat(1000)
    rewardbench   = rewardbench.select_columns(["text"]).repeat(1000)
    sycophancy    = sycophancy.select_columns(["text"]).repeat(1000)
    ultrafeedback = ultrafeedback.select_columns(["text"]).repeat(1000)

    # Sanity check — weights must sum to 1.0
    total = pile_weight + rewardbench_weight + sycophancy_weight + ultrafeedback_weight
    assert abs(total - 1.0) < 1e-6, f"Weights must sum to 1.0, got {total}"

    combined = interleave_datasets(
        [pile, rewardbench, sycophancy, ultrafeedback],
        probabilities=[pile_weight, rewardbench_weight,
                       sycophancy_weight, ultrafeedback_weight],
        seed=seed,
        stopping_strategy="all_exhausted"
    )

    return combined

In [5]:
import time
from requests.exceptions import ChunkedEncodingError
class MultiDatasetActivationBuffer:
    def __init__(
        self,
        model,
        hook_point,
        buffer_size=2**18,      # ~262K tokens live in GPU memory
        batch_size=4096,
        max_seq_len=512,        # truncate long docs to save time
        device="cuda",
        pile_weight=0.50,
        rewardbench_weight=0.20,
        sycophancy_weight=0.15,
        ultrafeedback_weight=0.15,
    ):
        self.model       = model
        self.hook_point  = hook_point
        self.buffer_size = buffer_size
        self.batch_size  = batch_size
        self.max_seq_len = max_seq_len
        self.device      = device

        self.buffer  = torch.zeros(
            buffer_size, model.cfg.d_model,
            dtype=torch.bfloat16, device=device  # bf16 halves memory
        )
        self.pointer = buffer_size  # start "empty" to trigger first fill
        # Add these inside __init__
        self._pile_weight        = pile_weight
        self._rewardbench_weight = rewardbench_weight
        self._sycophancy_weight  = sycophancy_weight
        self._ultrafeedback_weight = ultrafeedback_weight

        # Build the combined streaming dataset
        self.dataset = iter(build_combined_dataset(
            pile_weight=pile_weight,
            rewardbench_weight=rewardbench_weight,
            sycophancy_weight=sycophancy_weight,
            ultrafeedback_weight=ultrafeedback_weight
        ))

        # Track token counts per source for logging
        self.tokens_seen = {"pile": 0, "rewardbench": 0, "sycophancy": 0, "ultrafeedback": 0}

    @torch.no_grad()

    def _fill_buffer(self, llm_batch_size=64):
        print("Refilling activation buffer...")
        self.pointer = 0
        filled = 0
        pending_texts = []

        def process_batch(texts):
            nonlocal filled
            tokens = self.model.tokenizer(
                texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=self.max_seq_len,
            ).to(self.device)

            _, cache = self.model.run_with_cache(
                tokens["input_ids"],
                names_filter=self.hook_point,
                return_type=None,
            )

            # acts: (batch, seq_len, d_model)
            acts = cache[self.hook_point].to(torch.bfloat16)

            for i in range(acts.shape[0]):
                # Use attention mask to get real sequence length
                seq_len = tokens["attention_mask"][i].sum().item()
                act = acts[i, :seq_len]  # (real_seq_len, d_model)
                space = self.buffer_size - filled
                n = min(seq_len, space)
                self.buffer[filled : filled + n] = act[:n]
                filled += n

        while filled < self.buffer_size:
            # Fetch sample with retry
            sample = None
            for attempt in range(10):
                try:
                    sample = next(self.dataset)
                    break
                except StopIteration:
                    break
                except Exception as e:
                    wait = 5 * (attempt + 1)
                    print(f"Stream error ({type(e).__name__}), retrying in {wait}s...")
                    time.sleep(wait)
                    self.dataset = iter(build_combined_dataset(
                        pile_weight=self._pile_weight,
                        rewardbench_weight=self._rewardbench_weight,
                        sycophancy_weight=self._sycophancy_weight,
                        ultrafeedback_weight=self._ultrafeedback_weight
                    ))

            if sample is None:
                break

            text = sample.get("text", "")
            if not text or len(text.strip()) < 20:
                continue

            pending_texts.append(text)

            # Process in batches of llm_batch_size
            if len(pending_texts) == llm_batch_size:
                try:
                    process_batch(pending_texts)
                except Exception as e:
                    print(f"Batch processing error: {e}, skipping batch.")
                pending_texts = []

        # Process any remaining texts
        if pending_texts:
            try:
                process_batch(pending_texts)
            except Exception as e:
                print(f"Final batch error: {e}, skipping.")



        # Shuffle so batches aren't sequentially correlated
            perm = torch.randperm(filled, device=self.device)
            self.buffer[:filled] = self.buffer[perm]
            self.total_filled = filled


    def next_batch(self) -> torch.Tensor:
        """
        Returns the next batch of activations.
        Automatically refills the buffer when exhausted.
        """
        if self.pointer + self.batch_size > self.buffer_size:
            self._fill_buffer()

        batch = self.buffer[self.pointer : self.pointer + self.batch_size]
        self.pointer += self.batch_size
        return batch.float()

In [6]:
'''class MultiDatasetActivationBuffer:
    def __init__(
        self,
        model,
        hook_point,
        buffer_size=2**18,      # ~262K tokens live in GPU memory
        batch_size=4096,
        max_seq_len=512,        # truncate long docs to save time
        device="cuda",
        pile_weight=0.7,
        rewardbench_weight=0.2,
        sycophancy_weight=0.1,
    ):
        self.model       = model
        self.hook_point  = hook_point
        self.buffer_size = buffer_size
        self.batch_size  = batch_size
        self.max_seq_len = max_seq_len
        self.device      = device

        self.buffer  = torch.zeros(
            buffer_size, model.cfg.d_model,
            dtype=torch.bfloat16, device=device  # bf16 halves memory
        )
        self.pointer = buffer_size  # start "empty" to trigger first fill

        # Build the combined streaming dataset
        self.dataset = iter(build_combined_dataset(
            pile_weight=pile_weight,
            rewardbench_weight=rewardbench_weight,
            sycophancy_weight=sycophancy_weight
        ))

        # Track token counts per source for logging
        self.tokens_seen = {"pile": 0, "rewardbench": 0, "sycophancy": 0}

    @torch.no_grad()
    def _fill_buffer(self):
        """Pull fresh activations from the LLM until buffer is full."""
        print("Refilling activation buffer...")
        self.pointer = 0
        filled = 0

        while filled < self.buffer_size:
            try:
                sample = next(self.dataset)
            except StopIteration:
                break  # should never happen with never_stop, but be safe

            text = sample["text"]
            if not text or len(text.strip()) < 20:
                continue  # skip empty/junk samples

            # Tokenize with truncation
            tokens = self.model.to_tokens(
                text,
                prepend_bos=True,
            )
            if tokens.shape[1] > self.max_seq_len:
                tokens = tokens[:, :self.max_seq_len]

            # Run LLM, cache only the target hook point
            _, cache = self.model.run_with_cache(
                tokens,
                names_filter=self.hook_point,
                return_type=None,
                prepend_bos=False  # already added above
            )

            # (1, seq_len, d_model) → (seq_len, d_model)
            acts = cache[self.hook_point].squeeze(0).to(torch.bfloat16)
            seq_len = acts.shape[0]

            # How many tokens can we still fit?
            space = self.buffer_size - filled
            n = min(seq_len, space)

            self.buffer[filled : filled + n] = acts[:n]
            filled += n

        # Shuffle so batches aren't sequentially correlated
        perm = torch.randperm(filled, device=self.device)
        self.buffer[:filled] = self.buffer[perm]
        self.total_filled = filled
        print(f"Buffer filled with {filled:,} tokens.")

    def next_batch(self) -> torch.Tensor:
        """
        Returns the next batch of activations.
        Automatically refills the buffer when exhausted.
        """
        if self.pointer + self.batch_size > self.buffer_size:
            self._fill_buffer()

        batch = self.buffer[self.pointer : self.pointer + self.batch_size]
        self.pointer += self.batch_size
        return batch.float()  # SAE trains in fp32'''

'class MultiDatasetActivationBuffer:\n    def __init__(\n        self,\n        model,\n        hook_point,\n        buffer_size=2**18,      # ~262K tokens live in GPU memory\n        batch_size=4096,\n        max_seq_len=512,        # truncate long docs to save time\n        device="cuda",\n        pile_weight=0.7,\n        rewardbench_weight=0.2,\n        sycophancy_weight=0.1,\n    ):\n        self.model       = model\n        self.hook_point  = hook_point\n        self.buffer_size = buffer_size\n        self.batch_size  = batch_size\n        self.max_seq_len = max_seq_len\n        self.device      = device\n\n        self.buffer  = torch.zeros(\n            buffer_size, model.cfg.d_model,\n            dtype=torch.bfloat16, device=device  # bf16 halves memory\n        )\n        self.pointer = buffer_size  # start "empty" to trigger first fill\n\n        # Build the combined streaming dataset\n        self.dataset = iter(build_combined_dataset(\n            pile_weight=pile_weight,\

In [7]:
import torch
from tqdm import tqdm
from pathlib import Path
Home_Dir = "/content/drive/MyDrive"
#Model_Name = "google/gemma-2-2b"
Model_Name = "google/gemma-2-2b-it"
def train_matryoshka_saes_streaming(
    loader,
    widths,
    l1_coeff=3e-4,
    base_lr=1e-3,
    epochs=1,
    device="cuda",
    save_dir=f"/content/drive/MyDrive/{Model_Name}_Activations/SAES",
    checkpoint_every=1000,
):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    saes = []

    # ---- infer input dimension safely ----
    first_batch = next(iter(loader))
    d_in = first_batch.shape[-1]
    print("Inferred d_in =", d_in)

    for level, d_hidden in enumerate(widths):
      final_path = save_dir / f"matryoshka_sae_{level}.pt"
      if final_path.exists():
          print(f"\n=== Level {level} already complete, loading ===")
          sae = torch.load(final_path, map_location=device,weights_only=False).to(torch.bfloat16)
          sae.eval()
          for p in sae.parameters():
              p.requires_grad_(False)
          saes.append(sae)
          continue

      print(f"\n=== Training SAE {level} (d_hidden={d_hidden}) ===")

      sae = SAE(
          d_in=d_in,
          d_hidden=d_hidden,
          l1_coeff=l1_coeff,
      ).to(device).to(torch.bfloat16)

      lr = base_lr / (2 ** level)

      checkpoint_path = save_dir / f"checkpoint_level{level}.pt"
      start_step = 0

      # Load checkpoint BEFORE torch.compile
      if checkpoint_path.exists():
          print(f"Found checkpoint for level {level}, loading...")
          ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
          sae.load_state_dict(ckpt["sae_state"])
          opt = torch.optim.AdamW(sae.parameters(), lr=lr)
          opt.load_state_dict(ckpt["opt_state"])
          start_step = ckpt["step"] + 1
          print(f"Resuming from step {start_step}")
      else:
          opt = torch.optim.AdamW(sae.parameters(), lr=lr)

      # Compile AFTER loading
      sae = torch.compile(sae)

      for epoch in range(epochs):
        loop = tqdm(
            loader,
            desc=f"SAE {level} | Epoch {epoch+1}",
            total=len(loader),
            initial=start_step,
        )

        for local_step, x in enumerate(loop):
            # Offset by start_step so step reflects true global position
            step = local_step + start_step

            # Stop when we hit total steps
            if step >= len(loader):
                break

            x = x.to(device, dtype=torch.bfloat16)

            with torch.no_grad():
                residual = x
                for prev_sae in saes:
                    prev_sae.eval()
                    x_hat_prev, _ = prev_sae(residual)
                    residual = residual - x_hat_prev
                residual = residual / (
                    residual.norm(dim=1, keepdim=True) + 1e-6
                )

            x_hat, z = sae(residual)
            loss = sae.loss(residual, x_hat, z)

            opt.zero_grad()
            loss.backward()
            opt.step()

            loop.set_postfix(
                loss=f"{loss.item():.4f}",
                lr=f"{lr:.1e}",
                step=step,
            )

            if step % checkpoint_every == 0 and step > start_step:
                sae_to_save = sae._orig_mod if hasattr(sae, '_orig_mod') else sae
                torch.save({
                    "sae_state": sae_to_save.state_dict(),
                    "opt_state": opt.state_dict(),
                    "step": step,
                    "level": level,
                }, checkpoint_path)
                import os; os.sync()
                print(f"Checkpointed level {level} at step {step}")

            if step % 5000 == 0 and step > start_step:
                from IPython.display import clear_output
                clear_output(wait=True)
                print(f"Level {level} | Step {step}/{len(loader)} | Loss {loss.item():.4f}")
          # Freeze + save
      sae.eval()
      for p in sae.parameters():
          p.requires_grad_(False)
      saes.append(sae)

      sae_to_save = sae._orig_mod if hasattr(sae, '_orig_mod') else sae
      torch.save(sae_to_save.cpu(), final_path)
      sae.to(device).to(torch.bfloat16)
      print(f"Level {level} complete. Saved to {final_path}")

      if checkpoint_path.exists():
          checkpoint_path.unlink()

    return saes

In [8]:
class BufferIterator:
    """Makes MultiDatasetActivationBuffer compatible with for-loop iteration."""
    def __init__(self, buffer, n_steps):
        self.buffer  = buffer
        self.n_steps = n_steps

    def __iter__(self):
        for _ in range(self.n_steps):
            yield self.buffer.next_batch()

    def __len__(self):
        return self.n_steps

In [ ]:
# @title
import torch

 # Also upgrade datasets as it's a dependency and used in the notebook

import transformer_lens

from transformer_lens import HookedTransformer

# Load frozen LLM
model = HookedTransformer.from_pretrained(
    Model_Name,
    dtype=torch.bfloat16
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

# Build buffer — mixing 70% Pile, 20% RewardBench, 10% Sycophancy
buffer = MultiDatasetActivationBuffer(
    model=model,
    hook_point="blocks.24.hook_resid_post",
    buffer_size=2**17,
    batch_size=8192,
    max_seq_len=512,
    pile_weight=0.50,
    rewardbench_weight=0.20,
    sycophancy_weight=0.15,
    ultrafeedback_weight=0.15,
)

# SAE training loop
# ── Wrap buffer as an iterator ────────────────────────────────────────────────
N_STEPS = 50_000
loader = BufferIterator(buffer, n_steps=N_STEPS)
d_in = 2304
# ── Train Matryoshka SAEs ─────────────────────────────────────────────────────
saes = train_matryoshka_saes_streaming(
    loader=loader,
    widths=[int(d_in*2), int(d_in*8), int(d_in*16), int(d_in*32)],   # 0.125x / 1x / 4x / 16x hidden dim
    l1_coeff=3e-4,
    base_lr=2e-4,                        # matches your original lr
    epochs=1,
    device="cuda",
    save_dir=f"/content/drive/MyDrive/{Model_Name}_Activations/SAES",
    checkpoint_every=1000,
)

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

Refilling activation buffer...


SAE 0 | Epoch 1:   3%|▎         | 1374/50000 [17:06<8:11:08,  1.65it/s, loss=0.0000, lr=2.0e-04, step=1374]

Refilling activation buffer...


SAE 0 | Epoch 1:   3%|▎         | 1376/50000 [17:18<12:17:18,  1.10it/s, loss=0.0000, lr=2.0e-04, step=1390]

Refilling activation buffer...


In [ ]:
exit()

In [ ]:
'''import time

# Define SAEs list with SAE 0 for timing
saes = [
    SAE(
        d_in=2560,
        d_hidden=320,
        l1_coeff=3e-4,
    ).to("cuda").to(torch.bfloat16)
]
saes[0] = torch.compile(saes[0])


# Warmup compile
_ = test_sae(x)
start = time.time()
buffer._fill_buffer()
fill_time = time.time() - start
print(f"Fill time: {fill_time:.1f}s")

# Test 2 — how long is one pure SAE training step (no refill)?
x = buffer.next_batch().to("cuda", dtype=torch.bfloat16)
start = time.time()
for _ in range(100):
    x_hat, z = saes[0](x)
    loss = saes[0].loss(x, x_hat, z)
    loss.backward()
sae_time = (time.time() - start) / 100
print(f"SAE step time: {sae_time*1000:.1f}ms")

# Test 3 — steps per fill
steps_per_fill = buffer.buffer_size // buffer.batch_size
print(f"Steps per fill: {steps_per_fill}")
print(f"Fill overhead per step: {fill_time/steps_per_fill:.2f}s")
print(f"Estimated total time at current speed: {(fill_time/steps_per_fill + sae_time) * 50000 / 3600:.1f} hours")'''

In [ ]:
import time
start = time.time()
buffer._fill_buffer(llm_batch_size=64)
print(f"Fill time: {time.time() - start:.1f}s")


In [ ]:
# Run this in a new cell — if it executes, kernel is alive
print("Kernel is alive")